In [1]:
import sys
import asyncio
import time
import os

import numpy as np

from lsst.ts import salobj

from lsst.ts.observatory.control.auxtel.atcs import ATCS
from lsst.ts.observatory.control.auxtel.latiss import LATISS

In [2]:
print(os.environ["OSPL_URI"])
print(os.environ["LSST_DDS_PARTITION_PREFIX"])

file:///home/craiglagegit/WORK/ts_ddsconfig/config/ospl-shmem.xml
summit


In [3]:
import logging
stream_handler = logging.StreamHandler(sys.stdout)
logger = logging.getLogger()
logger.addHandler(stream_handler)
logger.level = logging.DEBUG

In [4]:
#Start classes
domain = salobj.Domain()
await asyncio.sleep(10) # This can be removed in the future...
atcs = ATCS(domain)
latiss = LATISS(domain)
await asyncio.gather(atcs.start_task, latiss.start_task)

atmcs: Adding all resources.
atptg: Adding all resources.
ataos: Adding all resources.
atpneumatics: Adding all resources.
athexapod: Adding all resources.
atdome: Adding all resources.
atdometrajectory: Adding all resources.
atcamera: Adding all resources.
atspectrograph: Adding all resources.
atheaderservice: Adding all resources.
atarchiver: Adding all resources.
Read historical data in 0.07 sec
Read 1 history items for RemoteEvent(ATDomeTrajectory, 0, authList)
Read 1 history items for RemoteEvent(ATDomeTrajectory, 0, followingMode)
Read 100 history items for RemoteEvent(ATDomeTrajectory, 0, heartbeat)
Read 1 history items for RemoteEvent(ATDomeTrajectory, 0, logLevel)
Read 2 history items for RemoteEvent(ATDomeTrajectory, 0, logMessage)
Read 1 history items for RemoteEvent(ATDomeTrajectory, 0, settingVersions)
Read 1 history items for RemoteEvent(ATDomeTrajectory, 0, simulationMode)
Read 1 history items for RemoteEvent(ATDomeTrajectory, 0, softwareVersions)
Read 1 history items fo

[[None, None, None, None, None, None, None], [None, None, None, None]]

In [6]:
# enable components
await atcs.enable({"atdome": "current", "ataos": "current", "athexapod": "current"})
await latiss.enable()

Enabling all components
Gathering settings.
Received settings from users.: {'atdome': 'current', 'ataos': 'current', 'athexapod': 'current'}
Couldn't get settingVersions event. Using empty settings.
Couldn't get settingVersions event. Using empty settings.
Couldn't get settingVersions event. Using empty settings.
Complete settings for atmcs.
Complete settings for atptg.
Complete settings for atpneumatics.
Complete settings for atdometrajectory.
Settings versions: {'atdome': 'current', 'ataos': 'current', 'athexapod': 'current', 'atmcs': '', 'atptg': '', 'atpneumatics': '', 'atdometrajectory': ''}
Unable to transition atmcs to <State.ENABLED: 2> NoneType: None
.
Traceback (most recent call last):
  File "/opt/lsst/src/ts_salobj/python/lsst/ts/salobj/csc_utils.py", line 129, in set_summary_state
    state_data = await remote.evt_summaryState.aget(timeout=timeout)
  File "/opt/lsst/src/ts_salobj/python/lsst/ts/salobj/topics/read_topic.py", line 495, in aget
    await asyncio.wait_for(self

RuntimeError: Failed to transition ['atmcs', 'atpneumatics'] to <State.ENABLED: 2>.

In [7]:
tmp=await latiss.rem.atheaderservice.cmd_start.set_start()
print(tmp)

private_revCode: 5907d2d0, private_sndStamp: 1620052611.6257849, private_rcvStamp: 1620052611.626286, private_seqNum: 2097192701, private_identity: ATHeaderService, private_origin: 259669, private_host: 0, ack: 303, error: 0, result: Done, host: 0, identity: craiglagegit@nb-craiglagegit-w-2021-18, origin: 8932, cmdtype: 10, timeout: 0.0


In [11]:
await salobj.set_summary_state(latiss.rem.atheaderservice, salobj.State.ENABLED)

[<State.ENABLED: 2>]

In [17]:
tmp=await atcs.rem.atmcs.cmd_start.set_start()
print(tmp)

CancelledError: 

In [18]:
await salobj.set_summary_state(atcs.rem.atmcs, salobj.State.STANDBY)

RuntimeError: Cannot get summaryState from ATMCS

In [13]:
tmp=await latiss.rem.atarchiver.cmd_start.set_start()
print(tmp)

AckError: msg='Command failed', ackcmd=(ackcmd private_seqNum=2044254915, ack=<SalRetCode.CMD_FAILED: -302>, error=1, result='Failed: start not allowed in state <State.DISABLED: 1>')

In [19]:
tmp = await atcs.rem.atmcs.evt_heartbeat.next(flush=True)
print(tmp)

CancelledError: 

In [20]:
await salobj.set_summary_state(latiss.rem.atarchiver, salobj.State.ENABLED)

RuntimeError: Error on cmd=cmd_enterControl, initial_state=4: msg='Command failed', ackcmd=(ackcmd private_seqNum=741305057, ack=<SalRetCode.CMD_FAILED: -302>, error=1, result='Failed: Not supported by this CSC')

In [ ]:
current_az = 120.0
current_el = 65.0
current_rot = -110.0
await atcs.point_azel(current_az, current_el, rot_tel=current_rot)

In [15]:
# Run 1 bias as a test
await latiss.take_bias(1)

Generating group_id
BIAS 0001 - 0001


array([2021031800001])

In [16]:
# Run 5 more
for i in range(5):
    await asyncio.sleep(2)
    await latiss.take_bias(1)

Generating group_id
BIAS 0001 - 0001
Generating group_id
BIAS 0001 - 0001
logMessage DDS read queue is filling: 16 of 100 elements
logMessage DDS read queue is filling: 21 of 100 elements
Generating group_id
BIAS 0001 - 0001
Generating group_id
BIAS 0001 - 0001
logMessage DDS read queue is filling: 10 of 100 elements
logMessage DDS read queue is filling: 12 of 100 elements
Generating group_id
BIAS 0001 - 0001


In [22]:
await latiss.standby()

Unable to transition atcamera to <State.STANDBY: 5> NoneType: None
.
Traceback (most recent call last):
  File "/opt/lsst/src/ts_salobj/python/lsst/ts/salobj/csc_utils.py", line 129, in set_summary_state
    state_data = await remote.evt_summaryState.aget(timeout=timeout)
  File "/opt/lsst/src/ts_salobj/python/lsst/ts/salobj/topics/read_topic.py", line 495, in aget
    await asyncio.wait_for(self._next_task, timeout=timeout)
  File "/opt/lsst/software/stack/conda/miniconda3-py38_4.9.2/envs/lsst-scipipe-0.5.0/lib/python3.8/asyncio/tasks.py", line 501, in wait_for
    raise exceptions.TimeoutError()
asyncio.exceptions.TimeoutError

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/opt/lsst/src/ts_salobj/python/lsst/ts/salobj/csc_utils.py", line 131, in set_summary_state
    raise RuntimeError(f"Cannot get summaryState from {remote.salinfo.name}")
RuntimeError: Cannot get summaryState from ATCamera

[atspectrograph]::[<State.S

RuntimeError: Failed to transition ['atcamera', 'atarchiver'] to <State.STANDBY: 5>.

In [23]:
await atcs.standby()

Unable to transition atmcs to <State.STANDBY: 5> NoneType: None
.
Traceback (most recent call last):
  File "/opt/lsst/src/ts_salobj/python/lsst/ts/salobj/csc_utils.py", line 129, in set_summary_state
    state_data = await remote.evt_summaryState.aget(timeout=timeout)
  File "/opt/lsst/src/ts_salobj/python/lsst/ts/salobj/topics/read_topic.py", line 495, in aget
    await asyncio.wait_for(self._next_task, timeout=timeout)
  File "/opt/lsst/software/stack/conda/miniconda3-py38_4.9.2/envs/lsst-scipipe-0.5.0/lib/python3.8/asyncio/tasks.py", line 501, in wait_for
    raise exceptions.TimeoutError()
asyncio.exceptions.TimeoutError

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/opt/lsst/src/ts_salobj/python/lsst/ts/salobj/csc_utils.py", line 131, in set_summary_state
    raise RuntimeError(f"Cannot get summaryState from {remote.salinfo.name}")
RuntimeError: Cannot get summaryState from ATMCS

[atptg]::[<State.STANDBY: 5>]
[at

RuntimeError: Failed to transition ['atmcs', 'atpneumatics'] to <State.STANDBY: 5>.